In [1]:
# confusion_matrix.ipynb - plot confusion matrix.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from logging import info, error
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix

In [3]:
root_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1r1n1t/vary_tumor_prop'
out_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1r1n1t/vary_tumor_prop/analysis/tumor_identification/calicost/confusion_matrix'

## Merge metrics from multiple experiments into one single file.

In [4]:
run_id_list = sorted([d for d in os.listdir(root_dir) if d.endswith('t')])
run_id_list

['100n900t',
 '10n990t',
 '300n700t',
 '30n970t',
 '500n500t',
 '700n300t',
 '900n100t',
 '970n30t',
 '990n10t',
 '997n3t']

In [5]:
def assert_e(path):
    """Assert file or folder exists, mimicking shell "test -e"."""
    assert path is not None and os.path.exists(path)

def plot_labels_confusion_matrix(
    run_id,
    tool_list,
    tool_fn_list,
    truth_fn,
    out_fig_fn,
    fig_width = None,
    fig_height = None,
    fig_nrow = None,
    fig_ncol = 3,
    fig_dpi = 300,
    verbose = True
):
    # check args.
    assert len(tool_list) == len(tool_fn_list)
    new_tool_list, new_tool_fn_list = [], []
    for tool, fn in zip(tool_list, tool_fn_list):
        if not os.path.exists(fn):
            print("W: '%s'-'%s' file does not exist!" % (run_id, tool))
        else:
            new_tool_list.append(tool)
            new_tool_fn_list.append(fn)
    tool_list, tool_fn_list = new_tool_list, new_tool_fn_list
    assert_e(truth_fn)


    # plot confusion matrix for each tool.
    if verbose:
        info("plot confusion matrix for each tool's labels ...")

    n = len(tool_list)
    fig_ncol = min(n, fig_ncol)
    if fig_nrow is None:
        fig_nrow = int(np.ceil(n / fig_ncol))
    if fig_width is None:
        fig_width = 2.5 * fig_ncol
    if fig_height is None:
        fig_height = 2.5 * fig_nrow
    
    # Create subplots with exact number needed (no empty subplots)
    fig, axes = plt.subplots(
        fig_nrow, fig_ncol,
        figsize = (fig_width, fig_height)
    )
    
    # Handle axes array - flatten if 2D, convert to list if single
    if n == 1:
        axes = [axes]
    elif fig_nrow == 1:
        axes = [axes] if not isinstance(axes, np.ndarray) else axes.tolist()
    else:
        axes = axes.flatten()
    
    # Hide unused subplots
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    # Publication-ready styling
    title_fontsize = 5
    label_fontsize = 5
    tick_fontsize = 5
    annot_fontsize = 6
    
    df = pd.read_csv(truth_fn, sep = '\t')
    truth_labels = df['annotation'].to_numpy()
    i = 0
    for ax, tool, tool_fn in zip(axes[:n], tool_list, tool_fn_list):
        display_name = tool
        if verbose:
            info("process %s ..." % display_name)

        df = pd.read_csv(tool_fn, sep = '\t')
        tool_labels = df['prediction'].to_numpy()
        print(len(truth_labels), len(tool_labels))
        cm = confusion_matrix(
            truth_labels, tool_labels,
            labels = ['normal', 'tumor']
        )
        if i == 0:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['Normal', 'Tumor'],
                yticklabels = ['Normal', 'Tumor'],
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )
        else:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['Normal', 'Tumor'],
                yticklabels = False,
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )            
        #ax.set_title(f'{display_name}', fontsize = title_fontsize, pad = 8)
        ax.set_title(f'{display_name}', fontsize = title_fontsize)
        ax.set_xlabel('Predicted', fontsize = label_fontsize)
        if i == 0:
            ax.set_ylabel('True', fontsize = label_fontsize)
        else:
            ax.set_ylabel(None)
        ax.tick_params(labelsize = label_fontsize)
        
        ax.tick_params(
            axis = 'x',          # Target x-axis ticks
            labelsize = tick_fontsize,
            rotation = 0         # Force horizontal (critical fix)
        )
        if i == 0:
            ax.tick_params(
                axis = 'y',          # Keep y-axis ticks as-is (vertical by default)
                labelsize = tick_fontsize
            )
        i += 1

    #plt.subplots_adjust(wspace = 0.1)
    plt.tight_layout()
    fig.savefig(out_fig_fn, dpi = fig_dpi, bbox_inches = 'tight')
    plt.close(fig)


    res = dict(
        out_fig_fn = out_fig_fn
    )
    return(res)

In [6]:
tool_list = ['ref(-)/purity(-)', 'ref(-)/purity(+)', 'ref(+)/purity(+)']
id_list = ['noref_nopurity', 'noref_purity', 'ref_purity']
for run_id in run_id_list:
    print("processing '%s' ..." % run_id)
    plot_labels_confusion_matrix(
        run_id = run_id,
        tool_list = tool_list,
        tool_fn_list = [os.path.join(root_dir, \
            '%s/tumor_identification/calicost/1_overlap/overlap.calicost_%s.tsv' % (run_id, x))   \
            for x in id_list],
        truth_fn = os.path.join(
            root_dir, '%s/tumor_identification/calicost/1_overlap/overlap.truth.tsv' % run_id),
        out_fig_fn = os.path.join(out_dir, '%s.png' % run_id),
        fig_width = 3,
        fig_height = 1.4,
        fig_nrow = 1,
        fig_ncol = 5,
        fig_dpi = 300,
        verbose = True
    )

processing '100n900t' ...
975 975
975 975
975 975
processing '10n990t' ...
W: '10n990t'-'ref(-)/purity(+)' file does not exist!
1000 1000
1000 1000
processing '300n700t' ...
999 999
999 999
999 999
processing '30n970t' ...
W: '30n970t'-'ref(-)/purity(+)' file does not exist!
999 999
999 999
processing '500n500t' ...
1000 1000
1000 1000
1000 1000
processing '700n300t' ...
1000 1000
1000 1000
1000 1000
processing '900n100t' ...
W: '900n100t'-'ref(-)/purity(+)' file does not exist!
W: '900n100t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '970n30t' ...
W: '970n30t'-'ref(-)/purity(+)' file does not exist!
W: '970n30t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '990n10t' ...
W: '990n10t'-'ref(-)/purity(+)' file does not exist!
W: '990n10t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '997n3t' ...
W: '997n3t'-'ref(-)/purity(+)' file does not exist!
W: '997n3t'-'ref(+)/purity(+)' file does not exist!
1000 1000


In [7]:
tool_list = ['ref(-)/purity(-)', 'ref(-)/purity(+)', 'ref(+)/purity(+)']
id_list = ['noref_nopurity', 'noref_purity', 'ref_purity']
for run_id in ['10n990t', '30n970t']:
    print("processing '%s' ..." % run_id)
    plot_labels_confusion_matrix(
        run_id = run_id,
        tool_list = tool_list,
        tool_fn_list = [os.path.join(root_dir, \
            '%s/tumor_identification/calicost/1_overlap/overlap.calicost_%s.tsv' % (run_id, x))   \
            for x in id_list],
        truth_fn = os.path.join(
            root_dir, '%s/tumor_identification/calicost/1_overlap/overlap.truth.tsv' % run_id),
        out_fig_fn = os.path.join(out_dir, '%s.png' % run_id),
        fig_width = 3*2/3 + 0.1,
        fig_height = 1.4,
        fig_nrow = 1,
        fig_ncol = 5,
        fig_dpi = 300,
        verbose = True
    )

processing '10n990t' ...
W: '10n990t'-'ref(-)/purity(+)' file does not exist!
1000 1000
1000 1000
processing '30n970t' ...
W: '30n970t'-'ref(-)/purity(+)' file does not exist!
999 999
999 999


In [8]:
tool_list = ['ref(-)/purity(-)', 'ref(-)/purity(+)', 'ref(+)/purity(+)']
id_list = ['noref_nopurity', 'noref_purity', 'ref_purity']
for run_id in ['900n100t', '970n30t', '990n10t', '997n3t']:
    print("processing '%s' ..." % run_id)
    plot_labels_confusion_matrix(
        run_id = run_id,
        tool_list = tool_list,
        tool_fn_list = [os.path.join(root_dir, \
            '%s/tumor_identification/calicost/1_overlap/overlap.calicost_%s.tsv' % (run_id, x))   \
            for x in id_list],
        truth_fn = os.path.join(
            root_dir, '%s/tumor_identification/calicost/1_overlap/overlap.truth.tsv' % run_id),
        out_fig_fn = os.path.join(out_dir, '%s.png' % run_id),
        fig_width = 3*1/3+0.3,
        fig_height = 1.4,
        fig_nrow = 1,
        fig_ncol = 5,
        fig_dpi = 300,
        verbose = True
    )

processing '900n100t' ...
W: '900n100t'-'ref(-)/purity(+)' file does not exist!
W: '900n100t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '970n30t' ...
W: '970n30t'-'ref(-)/purity(+)' file does not exist!
W: '970n30t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '990n10t' ...
W: '990n10t'-'ref(-)/purity(+)' file does not exist!
W: '990n10t'-'ref(+)/purity(+)' file does not exist!
1000 1000
processing '997n3t' ...
W: '997n3t'-'ref(-)/purity(+)' file does not exist!
W: '997n3t'-'ref(+)/purity(+)' file does not exist!
1000 1000
